In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px

Azure

In [3]:
azure_url = "https://learn.microsoft.com/en-us/azure/reliability/regions-list"

tables = pd.read_html(azure_url)

len(tables)

7

In [4]:
for i, table in enumerate(tables):
    print("Table", i)
    print(table.head())
    print(table.columns)
    print("-" * 80)

Table 0
   Symbol                                        Description
0     NaN  Region coming soon. To learn more about availa...
1     NaN  Access to this region is restricted to support...
Index(['Symbol', 'Description'], dtype='object')
--------------------------------------------------------------------------------
Table 1
                Region  Availability zone support        Paired region  \
0    Australia Central                        NaN  Australia Central 2   
1  Australia Central 2                        NaN    Australia Central   
2       Australia East                        NaN  Australia Southeast   
3  Australia Southeast                        NaN       Australia East   
4         Austria East                        NaN                  NaN   

  Physical location  Geography   Programmatic name  
0          Canberra  Australia    australiacentral  
1          Canberra  Australia   australiacentral2  
2   New South Wales  Australia       australiaeast  
3          Vic

In [5]:
azure_locations = pd.concat(
    tables[1:],
    ignore_index=True
)

azure_locations.shape

(114, 6)

In [6]:
azure_locations.head()

,Region,Availability zone support,Paired region,Physical location,Geography,Programmatic name
0,Australia Central,NaN,Australia Central 2,Canberra,Australia,australiacentral
1,Australia Central 2,NaN,Australia Central,Canberra,Australia,australiacentral2
2,Australia East,NaN,Australia Southeast,New South Wales,Australia,australiaeast
3,Australia Southeast,NaN,Australia East,Victoria,Australia,australiasoutheast
4,Austria East,NaN,NaN,Vienna,Austria,austriaeast


In [7]:
azure_locations = azure_locations[
    [
        "Region",
        "Physical location",
        "Geography",
        "Programmatic name"
    ]
].copy()

In [8]:
azure_locations.columns = [
    "region_name",
    "city",
    "country",
    "region_code"
]

In [9]:
azure_locations["provider"] = "Azure"

In [10]:
azure_locations = azure_locations[
    [
        "provider",
        "region_code",
        "region_name",
        "city",
        "country"
    ]
]

In [11]:
azure_locations.head(20)

,provider,region_code,region_name,city,country
0,Azure,australiacentral,Australia Central,Canberra,Australia
1,Azure,australiacentral2,Australia Central 2,Canberra,Australia
2,Azure,australiaeast,Australia East,New South Wales,Australia
3,Azure,australiasoutheast,Australia Southeast,Victoria,Australia
4,Azure,austriaeast,Austria East,Vienna,Austria
5,Azure,belgiumcentral,Belgium Central,Brussels,Belgium
6,Azure,brazilsouth,Brazil South,Sao Paulo State,Brazil
7,Azure,brazilsoutheast,Brazil Southeast,Rio,Brazil
8,Azure,canadacentral,Canada Central,Toronto,Canada
9,Azure,canadaeast,Canada East,Quebec,Canada


In [12]:
azure_locations.to_csv(
    "azure_locations.csv",
    index=False
)

In [14]:
azure_locations.shape

(114, 5)

In [15]:
azure_locations.head()

,provider,region_code,region_name,city,country
0,Azure,australiacentral,Australia Central,Canberra,Australia
1,Azure,australiacentral2,Australia Central 2,Canberra,Australia
2,Azure,australiaeast,Australia East,New South Wales,Australia
3,Azure,australiasoutheast,Australia Southeast,Victoria,Australia
4,Azure,austriaeast,Austria East,Vienna,Austria


In [16]:
azure_locations["country"].nunique()

30

In [17]:
azure_locations["country"].value_counts().head(20)

country
United States     18
Australia          8
India              8
Canada             4
Asia Pacific       4
Brazil             4
Germany            4
France             4
UAE                4
Switzerland        4
Norway             4
Europe             4
Korea              4
Japan              4
United Kingdom     4
South Africa       4
Israel             2
Indonesia          2
Austria            2
Belgium            2
Name: count, dtype: int64

In [18]:
azure_locations.to_csv(
    "azure_locations.csv",
    index=False
)

print("saved")

saved


AWS

In [19]:
import requests
import pandas as pd

In [21]:
import requests
import pandas as pd

url = "https://raw.githubusercontent.com/boto/botocore/develop/botocore/data/endpoints.json"

r = requests.get(url)

print(r.status_code)
print(r.text[:100])

200
{
  "partitions" : [ {
    "defaults" : {
      "hostname" : "{service}.{region}.{dnsSuffix}",
     


In [22]:
data = r.json()

list(data.keys())

['partitions', 'version']

In [23]:
aws_regions = []

for partition in data["partitions"]:
    if partition["partition"] != "aws":
        continue

    for region_code, info in partition["regions"].items():
        aws_regions.append({
            "provider": "AWS",
            "region_code": region_code,
            "region_name": info.get("description")
        })

aws_regions_df = pd.DataFrame(aws_regions)

aws_regions_df.shape

(34, 3)

In [24]:
aws_regions_df.head(20)

,provider,region_code,region_name
0,AWS,af-south-1,Africa (Cape Town)
1,AWS,ap-east-1,Asia Pacific (Hong Kong)
2,AWS,ap-east-2,Asia Pacific (Taipei)
3,AWS,ap-northeast-1,Asia Pacific (Tokyo)
4,AWS,ap-northeast-2,Asia Pacific (Seoul)
5,AWS,ap-northeast-3,Asia Pacific (Osaka)
6,AWS,ap-south-1,Asia Pacific (Mumbai)
7,AWS,ap-south-2,Asia Pacific (Hyderabad)
8,AWS,ap-southeast-1,Asia Pacific (Singapore)
9,AWS,ap-southeast-2,Asia Pacific (Sydney)


In [27]:
import re

def parse_aws_location(region_name):

    match = re.search(r"\((.*?)\)", region_name)

    if match:
        city = match.group(1)
    else:
        city = None

    continent = region_name.split("(")[0].strip()

    return pd.Series([city, continent])

aws_regions_df[
    ["city", "continent"]
] = aws_regions_df["region_name"].apply(
    parse_aws_location
)

In [28]:
aws_regions_df.head(20)

,provider,region_code,region_name,city,continent
0,AWS,af-south-1,Africa (Cape Town),Cape Town,Africa
1,AWS,ap-east-1,Asia Pacific (Hong Kong),Hong Kong,Asia Pacific
2,AWS,ap-east-2,Asia Pacific (Taipei),Taipei,Asia Pacific
3,AWS,ap-northeast-1,Asia Pacific (Tokyo),Tokyo,Asia Pacific
4,AWS,ap-northeast-2,Asia Pacific (Seoul),Seoul,Asia Pacific
5,AWS,ap-northeast-3,Asia Pacific (Osaka),Osaka,Asia Pacific
6,AWS,ap-south-1,Asia Pacific (Mumbai),Mumbai,Asia Pacific
7,AWS,ap-south-2,Asia Pacific (Hyderabad),Hyderabad,Asia Pacific
8,AWS,ap-southeast-1,Asia Pacific (Singapore),Singapore,Asia Pacific
9,AWS,ap-southeast-2,Asia Pacific (Sydney),Sydney,Asia Pacific


In [29]:
aws_country_map = {
    "Cape Town":"South Africa",
    "Hong Kong":"Hong Kong",
    "Taipei":"Taiwan",
    "Tokyo":"Japan",
    "Seoul":"South Korea",
    "Osaka":"Japan",
    "Mumbai":"India",
    "Hyderabad":"India",
    "Singapore":"Singapore",
    "Sydney":"Australia",
    "Jakarta":"Indonesia",
    "Melbourne":"Australia",
    "Malaysia":"Malaysia",
    "New Zealand":"New Zealand",
    "Thailand":"Thailand",
    "Central":"Canada",
    "Calgary":"Canada",
    "Frankfurt":"Germany",
    "Zurich":"Switzerland",
    "Stockholm":"Sweden",
    "Milan":"Italy",
    "Spain":"Spain",
    "Paris":"France",
    "Ireland":"Ireland",
    "London":"United Kingdom",
    "Tel Aviv":"Israel",
    "Bahrain":"Bahrain",
    "UAE":"United Arab Emirates",
    "N. Virginia":"United States",
    "Ohio":"United States",
    "N. California":"United States",
    "Oregon":"United States",
    "Mexico":"Mexico",
    "Sao Paulo":"Brazil"
}

In [30]:
aws_regions_df["country"] = (
    aws_regions_df["city"]
    .map(aws_country_map)
)

In [31]:
aws_regions_df[
    aws_regions_df["country"].isna()
]

,provider,region_code,region_name,city,continent,country


In [32]:
aws_regions_df.to_csv(
    "aws_locations.csv",
    index=False
)

aws_regions_df.shape

(34, 6)

In [33]:
aws_regions_df["region_code"].tolist()

['af-south-1',
 'ap-east-1',
 'ap-east-2',
 'ap-northeast-1',
 'ap-northeast-2',
 'ap-northeast-3',
 'ap-south-1',
 'ap-south-2',
 'ap-southeast-1',
 'ap-southeast-2',
 'ap-southeast-3',
 'ap-southeast-4',
 'ap-southeast-5',
 'ap-southeast-6',
 'ap-southeast-7',
 'ca-central-1',
 'ca-west-1',
 'eu-central-1',
 'eu-central-2',
 'eu-north-1',
 'eu-south-1',
 'eu-south-2',
 'eu-west-1',
 'eu-west-2',
 'eu-west-3',
 'il-central-1',
 'me-central-1',
 'me-south-1',
 'mx-central-1',
 'sa-east-1',
 'us-east-1',
 'us-east-2',
 'us-west-1',
 'us-west-2']

GCP

In [34]:
import pandas as pd

tables = pd.read_html(
    "https://cloud.google.com/compute/docs/regions-zones"
)

len(tables)

1

In [35]:
gcp_table = tables[0].copy()

gcp_table.head(20)

,Zones,Location,Machine types,CPUs,Options,CO2 emissions
0,africa-south1-a,"Johannesburg, South Africa","E2, N4&hairsp;, N2&hairsp;, N2D, C4&hairsp;, C...","Intel Cascade Lake, Ice Lake, Emerald Rapids, ...",AMD SEV,NaN
1,africa-south1-b,"Johannesburg, South Africa","E2, N4&hairsp;, N2&hairsp;, N2D, C4&hairsp;, T2D","Intel Cascade Lake, Ice Lake, Emerald Rapids, ...",AMD SEV,NaN
2,africa-south1-c,"Johannesburg, South Africa","E2, N4&hairsp;, N2&hairsp;, N2D, C4&hairsp;, T...","Intel Cascade Lake, Ice Lake, Emerald Rapids, ...",AMD SEV,NaN
3,asia-east1-a,"Changhua County, Taiwan, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, C4&hairsp...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN
4,asia-east1-b,"Changhua County, Taiwan, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, C4&hairsp...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN
5,asia-east1-c,"Changhua County, Taiwan, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, C4&hairsp...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN
6,asia-east2-a,"Hong Kong, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, C4&hairsp...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN
7,asia-east2-b,"Hong Kong, APAC","E2, N2&hairsp;, N2D, N1, T2D, C4&hairsp;, C2&h...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...",AMD SEV,NaN
8,asia-east2-c,"Hong Kong, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, T2D, C4&h...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN
9,asia-northeast1-a,"Tokyo, Japan, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, T2D, Z3, ...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN


In [36]:
gcp_table.columns

Index(['Zones', 'Location', 'Machine types', 'CPUs', 'Options',
       'CO2 emissions'],
      dtype='object')

In [37]:
gcp_locations = gcp_table.copy()

gcp_locations["region_code"] = (
    gcp_locations["Zones"]
    .str[:-2]
)

gcp_locations.head()

,Zones,Location,Machine types,CPUs,Options,CO2 emissions,region_code
0,africa-south1-a,"Johannesburg, South Africa","E2, N4&hairsp;, N2&hairsp;, N2D, C4&hairsp;, C...","Intel Cascade Lake, Ice Lake, Emerald Rapids, ...",AMD SEV,NaN,africa-south1
1,africa-south1-b,"Johannesburg, South Africa","E2, N4&hairsp;, N2&hairsp;, N2D, C4&hairsp;, T2D","Intel Cascade Lake, Ice Lake, Emerald Rapids, ...",AMD SEV,NaN,africa-south1
2,africa-south1-c,"Johannesburg, South Africa","E2, N4&hairsp;, N2&hairsp;, N2D, C4&hairsp;, T...","Intel Cascade Lake, Ice Lake, Emerald Rapids, ...",AMD SEV,NaN,africa-south1
3,asia-east1-a,"Changhua County, Taiwan, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, C4&hairsp...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN,asia-east1
4,asia-east1-b,"Changhua County, Taiwan, APAC","E2, N4&hairsp;, N2&hairsp;, N2D, N1, C4&hairsp...","Intel Ivy Bridge, Sandy Bridge, Haswell, Broad...","GPUs, AMD SEV",NaN,asia-east1


In [38]:
gcp_locations = gcp_locations.drop_duplicates(
    subset=["region_code"]
)

gcp_locations.shape

(43, 7)

In [39]:
def parse_location(x):

    parts = [p.strip() for p in str(x).split(",")]

    city = parts[0]

    if len(parts) >= 2:
        country = parts[1]
    else:
        country = None

    if len(parts) >= 3:
        continent = parts[2]
    else:
        continent = None

    return pd.Series([city, country, continent])

gcp_locations[
    ["city", "country", "continent"]
] = gcp_locations["Location"].apply(
    parse_location
)

In [40]:
gcp_locations["provider"] = "GCP"

gcp_locations["region_name"] = (
    gcp_locations["region_code"]
)

In [41]:
gcp_locations = gcp_locations[
    [
        "provider",
        "region_code",
        "region_name",
        "city",
        "country",
        "continent"
    ]
]

In [42]:
gcp_locations.head(20)

,provider,region_code,region_name,city,country,continent
0,GCP,africa-south1,africa-south1,Johannesburg,South Africa,None
3,GCP,asia-east1,asia-east1,Changhua County,Taiwan,APAC
6,GCP,asia-east2,asia-east2,Hong Kong,APAC,None
9,GCP,asia-northeast1,asia-northeast1,Tokyo,Japan,APAC
12,GCP,asia-northeast2,asia-northeast2,Osaka,Japan,APAC
15,GCP,asia-northeast3,asia-northeast3,Seoul,South Korea,APAC
18,GCP,asia-south1,asia-south1,Mumbai,India,APAC
21,GCP,asia-south2,asia-south2,Delhi,India,APAC
24,GCP,asia-southeast1,asia-southeast1,Jurong West,Singapore,APAC
27,GCP,asia-southeast2,asia-southeast2,Jakarta,Indonesia,APAC


In [43]:
gcp_locations.shape

(43, 6)

In [44]:
gcp_locations.head()

,provider,region_code,region_name,city,country,continent
0,GCP,africa-south1,africa-south1,Johannesburg,South Africa,None
3,GCP,asia-east1,asia-east1,Changhua County,Taiwan,APAC
6,GCP,asia-east2,asia-east2,Hong Kong,APAC,None
9,GCP,asia-northeast1,asia-northeast1,Tokyo,Japan,APAC
12,GCP,asia-northeast2,asia-northeast2,Osaka,Japan,APAC


In [45]:
def fix_gcp_country_continent(row):
    city = row["city"]
    country = row["country"]
    continent = row["continent"]

    if city == "Hong Kong":
        country = "Hong Kong"
        continent = "APAC"

    if city == "Johannesburg":
        country = "South Africa"
        continent = "Africa"

    if continent is None:
        continent = "Unknown"

    return pd.Series([country, continent])

gcp_locations[["country", "continent"]] = gcp_locations.apply(
    fix_gcp_country_continent,
    axis=1
)

In [46]:
gcp_locations[gcp_locations["continent"] == "Unknown"]

,provider,region_code,region_name,city,country,continent


In [47]:
gcp_locations.to_csv(
    "gcp_locations.csv",
    index=False
)

In [48]:
cloud_locations = pd.concat(
    [
        aws_regions_df,
        azure_locations,
        gcp_locations
    ],
    ignore_index=True
)

cloud_locations.shape

(191, 6)

In [49]:
cloud_locations.to_csv(
    "cloud_locations_dataset.csv",
    index=False
)

In [50]:
gcp_locations.isna().sum()

provider       0
region_code    0
region_name    0
city           0
country        0
continent      0
dtype: int64

In [51]:
cloud_locations = pd.concat(
    [
        aws_regions_df,
        azure_locations,
        gcp_locations
    ],
    ignore_index=True
)

cloud_locations.shape

(191, 6)

In [52]:
cloud_locations["provider"].value_counts()

provider
Azure    114
GCP       43
AWS       34
Name: count, dtype: int64

In [53]:
cloud_locations.to_csv(
    "cloud_locations_dataset.csv",
    index=False
)

In [54]:
region_summary = (
    cloud_locations
    .groupby("provider")
    .size()
    .reset_index(name="region_count")
)

region_summary

,provider,region_count
0,AWS,34
1,Azure,114
2,GCP,43


In [55]:
region_summary.to_csv(
    "region_summary.csv",
    index=False
)

In [57]:
country_summary = (
    cloud_locations
    .groupby("provider")["country"]
    .nunique()
    .reset_index(name="country_count")
)

country_summary

,provider,country_count
0,AWS,26
1,Azure,30
2,GCP,38


In [58]:
country_summary.to_csv(
    "country_coverage.csv",
    index=False
)